# 005. Open data is not licence-free: reading the OpenAQ /licenses endpoint

Code from post 005 of the 365 days of Microsoft Fabric series, in the order the post runs it. The post explains each cell. The post publishes on t-sql.dk on Friday 13 November 2026.

*Status: runnable design. The license fields below have been confirmed against the live API. The notebook has not been run in my workspace yet. Its fixture test and the five governance checks run before a report relies on the result, and their output goes into this post before it is published.*

In [ ]:
%pip install semantic-link-labs

In [ ]:
import json
import time
from datetime import date, datetime, timezone
import requests
from pyspark.sql import functions as F

VAULT_URL = "https://<your-vault-name>.vault.azure.net/"
LOCATION_IDS = [2993, 10516]
MODEL_NAME = "OpenAQ air quality"

API = "https://api.openaq.org/v3"
session = requests.Session()
session.headers["X-API-Key"] = notebookutils.credentials.getSecret(VAULT_URL, "openaq-api-key")

def openaq_get(path):
    response = session.get(f"{API}{path}", timeout=30)
    if response.status_code == 429:
        time.sleep(int(response.headers.get("x-ratelimit-reset", "60")) + 1)
        response = session.get(f"{API}{path}", timeout=30)
    response.raise_for_status()
    time.sleep(1)
    return response.json()["results"]

retrieved_at = datetime.now(timezone.utc).replace(tzinfo=None)
locations = [openaq_get(f"/locations/{location_id}")[0] for location_id in LOCATION_IDS]
license_ids = sorted({lic["id"] for loc in locations for lic in (loc["licenses"] or [])})
terms = {license_id: openaq_get(f"/licenses/{license_id}")[0] for license_id in license_ids}

stamp = retrieved_at.strftime("%Y%m%dT%H%M%SZ")
notebookutils.fs.put(f"Files/landing/openaq/metadata/locations_{stamp}.json", json.dumps(locations), True)
notebookutils.fs.put(f"Files/landing/openaq/metadata/licenses_{stamp}.json", json.dumps(terms), True)

In [ ]:
PERIOD_SCHEMA = """
    LocationId bigint, LicenseId bigint, LicenseName string, AttributionName string,
    AttributionUrl string, ValidFrom date, ValidTo date, CommercialUseAllowed boolean,
    AttributionRequired boolean, ShareAlikeRequired boolean, ModificationAllowed boolean,
    RedistributionAllowed boolean, SourceUrl string, RetrievedAtUtc timestamp
"""

def as_date(value):
    return date.fromisoformat(value[:10]) if value else None

periods = []
for loc in locations:
    for lic in loc["licenses"] or []:
        attribution = lic.get("attribution") or {}
        term = terms[lic["id"]]
        periods.append((
            loc["id"], lic["id"], lic["name"], attribution.get("name"), attribution.get("url"),
            as_date(lic.get("dateFrom")), as_date(lic.get("dateTo")),
            term["commercialUseAllowed"], term["attributionRequired"], term["shareAlikeRequired"],
            term["modificationAllowed"], term["redistributionAllowed"], term["sourceUrl"],
            retrieved_at,
        ))

period_df = spark.createDataFrame(periods, PERIOD_SCHEMA)
dim_license = (period_df
    .withColumn("LocationLicenseKey", F.xxhash64("LocationId", "LicenseId", "ValidFrom"))
    .select("LocationLicenseKey", *period_df.columns))
unknown = spark.createDataFrame(
    [(-1, None, None, "No license period found", None, None, None, None,
      None, None, None, None, None, None, retrieved_at)],
    dim_license.schema,
)
dim_license = dim_license.unionByName(unknown)

In [ ]:
def assign_license(fact_df, periods_df):
    """Attach the license period covering each reading's local date. No match gives -1."""
    measured = F.to_date(F.col("f.DateKey").cast("string"), "yyyyMMdd")
    return (fact_df.drop("LocationLicenseKey").alias("f")
        .join(periods_df.alias("p"),
              (F.col("f.LocationId") == F.col("p.LocationId"))
              & (measured >= F.col("p.ValidFrom"))
              & (F.col("p.ValidTo").isNull() | (measured <= F.col("p.ValidTo"))),
              "left")
        .select("f.*", F.coalesce(F.col("p.LocationLicenseKey"), F.lit(-1).cast("bigint"))
                .alias("LocationLicenseKey")))

In [ ]:
fixture_periods = spark.createDataFrame(
    [(1, 99, date(2026, 1, 1), date(2026, 9, 15)),
     (2, 99, date(2026, 9, 16), None)],
    "LocationLicenseKey bigint, LocationId bigint, ValidFrom date, ValidTo date",
)
fixture_fact = spark.createDataFrame(
    [(99, 20260915), (99, 20260916), (98, 20260916)],
    "LocationId bigint, DateKey int",
)
got = sorted(tuple(r) for r in assign_license(fixture_fact, fixture_periods)
             .select("LocationId", "DateKey", "LocationLicenseKey").collect())
expected = [(98, 20260916, -1), (99, 20260915, 1), (99, 20260916, 2)]
assert got == expected, got
print("Fixture passed: last day of a period, first day of the next, and no period at all")

In [ ]:
dim_license.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("dim_location_license")

fact = spark.table("fact_measurement")
fact_with_license = assign_license(fact, spark.table("dim_location_license")).localCheckpoint()
assert fact_with_license.count() == fact.count(), "Overlapping license periods duplicated readings"
fact_with_license.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("fact_measurement")

In [ ]:
from sempy_labs import refresh_semantic_model
from sempy_labs.directlake import add_table_to_direct_lake_semantic_model
from sempy_labs.tom import connect_semantic_model

add_table_to_direct_lake_semantic_model(
    dataset=MODEL_NAME, table_name="dim_location_license",
    lakehouse_table_name="dim_location_license", refresh=False,
)
with connect_semantic_model(dataset=MODEL_NAME, readonly=False) as tom:
    tom.add_data_column(
        table_name="fact_measurement", column_name="LocationLicenseKey",
        source_column="LocationLicenseKey", data_type="Int64", hidden=True,
    )
    tom.add_relationship(
        from_table="fact_measurement", from_column="LocationLicenseKey",
        to_table="dim_location_license", to_column="LocationLicenseKey",
        from_cardinality="Many", to_cardinality="One", cross_filtering_behavior="OneDirection",
    )
refresh_semantic_model(dataset=MODEL_NAME)

In [ ]:
CHECKS = {
    "Locations in the fact without a provider": """
        SELECT DISTINCT l.LocationId, l.LocationName
        FROM fact_measurement f JOIN dim_location l ON l.LocationId = f.LocationId
        WHERE l.ProviderName IS NULL OR trim(l.ProviderName) = ''""",
    "Readings outside every known license period": """
        SELECT LocationId, COUNT(*) AS Readings
        FROM fact_measurement WHERE LocationLicenseKey = -1
        GROUP BY LocationId""",
    "Attribution required but no attribution name": """
        SELECT DISTINCT d.LocationId, d.LicenseName
        FROM fact_measurement f JOIN dim_location_license d ON d.LocationLicenseKey = f.LocationLicenseKey
        WHERE d.AttributionRequired AND (d.AttributionName IS NULL OR trim(d.AttributionName) = '')""",
    "License periods in use without a source URL": """
        SELECT DISTINCT d.LocationId, d.LicenseName
        FROM fact_measurement f JOIN dim_location_license d ON d.LocationLicenseKey = f.LocationLicenseKey
        WHERE d.LocationLicenseKey <> -1 AND (d.SourceUrl IS NULL OR trim(d.SourceUrl) = '')""",
}
for name, query in CHECKS.items():
    rows = spark.sql(query).collect()
    print(f"{'OK  ' if not rows else 'FAIL'} {name}: {len(rows)}")
    for row in rows:
        print("     ", row.asDict())

In [ ]:
contributors = spark.sql("""
    SELECT DISTINCT l.ProviderName, d.AttributionName, d.LicenseName, d.SourceUrl
    FROM fact_measurement f
    JOIN dim_location l ON l.LocationId = f.LocationId
    JOIN dim_location_license d ON d.LocationLicenseKey = f.LocationLicenseKey
    WHERE d.LocationLicenseKey <> -1
    ORDER BY l.ProviderName
""").collect()
lines = ["Data: OpenAQ (openaq.org)."] + [
    f"{c.ProviderName}: {c.AttributionName or c.ProviderName}, {c.LicenseName} ({c.SourceUrl})"
    for c in contributors
]
print("\n".join(lines))